# Social Media Models — Inference

Loads both social media models and writes predictions to `operational.social_media_predictions`.

| Column | Description |
|---|---|
| `predicted_engagement_tier` | Low / Medium / High |
| `prob_engagement_low` – `prob_engagement_high` | Per-tier probabilities |
| `predicted_has_donations` | 0 / 1 |
| `prob_has_donations` | Probability that the post led to donations |

In [ ]:
import sys
sys.path.insert(0, '../jobs')

import json
from datetime import datetime, timezone

import joblib
import pandas as pd

from config import (
    DONATION_METADATA_PATH, DONATION_MODEL_PATH,
    ENGAGEMENT_METADATA_PATH, ENGAGEMENT_MODEL_PATH,
    OPERATIONAL_SCHEMA,
)
from utils_db import ensure_social_predictions_table, get_engine, pg_conn

print('Imports loaded.')

## Load models

In [ ]:
engagement_model = joblib.load(str(ENGAGEMENT_MODEL_PATH))
donation_model   = joblib.load(str(DONATION_MODEL_PATH))

with open(ENGAGEMENT_METADATA_PATH) as f: eng_meta = json.load(f)
with open(DONATION_METADATA_PATH)   as f: don_meta = json.load(f)

eng_classes = eng_meta['classes']   # e.g. ['High', 'Low', 'Medium']
don_classes = don_meta['classes']   # [0, 1]

print(f"Engagement model v{eng_meta['model_version']} | classes: {eng_classes}")
print(f"Donation model   v{don_meta['model_version']} | classes: {don_classes}")

## Build features from operational schema

In [ ]:
from inference_social import _build_features

engine = get_engine(OPERATIONAL_SCHEMA)
df = _build_features(engine)
print(f'Scoring {len(df)} posts')
df.head()

## Run predictions

In [ ]:
feature_cols = eng_meta['feature_cols']
X = df[feature_cols]

# Engagement predictions
eng_probs   = engagement_model.predict_proba(X)
eng_preds   = engagement_model.predict(X)
eng_prob_df = pd.DataFrame(eng_probs, columns=eng_classes)

# Donation predictions — prob column at index of class 1 = P(has_donations=True)
don_probs           = donation_model.predict_proba(X)
don_preds           = donation_model.predict(X)
don_class_idx       = list(don_classes).index(1) if 1 in don_classes else 1
prob_has_donations  = don_probs[:, don_class_idx]

ts = datetime.now(timezone.utc).isoformat()

rows = []
for i, (_, row) in enumerate(df.iterrows()):
    prob_low    = float(eng_prob_df.iloc[i]['Low'])    if 'Low'    in eng_prob_df.columns else None
    prob_medium = float(eng_prob_df.iloc[i]['Medium']) if 'Medium' in eng_prob_df.columns else None
    prob_high   = float(eng_prob_df.iloc[i]['High'])   if 'High'   in eng_prob_df.columns else None
    rows.append((
        int(row['post_id']),
        str(eng_preds[i]),
        prob_low,
        prob_medium,
        prob_high,
        int(don_preds[i]),
        float(prob_has_donations[i]),
        ts,
    ))

print(f'Built {len(rows)} prediction rows')

## Write to database

In [ ]:
ensure_social_predictions_table(OPERATIONAL_SCHEMA)

with pg_conn(OPERATIONAL_SCHEMA) as conn:
    with conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO operational.social_media_predictions
                (post_id, predicted_engagement_tier,
                 prob_engagement_low, prob_engagement_medium, prob_engagement_high,
                 predicted_has_donations, prob_has_donations, prediction_ts)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (post_id) DO UPDATE SET
                predicted_engagement_tier = EXCLUDED.predicted_engagement_tier,
                prob_engagement_low       = EXCLUDED.prob_engagement_low,
                prob_engagement_medium    = EXCLUDED.prob_engagement_medium,
                prob_engagement_high      = EXCLUDED.prob_engagement_high,
                predicted_has_donations   = EXCLUDED.predicted_has_donations,
                prob_has_donations        = EXCLUDED.prob_has_donations,
                prediction_ts             = EXCLUDED.prediction_ts
        """, rows)

print(f'Wrote {len(rows)} predictions to social_media_predictions')
print(f'Timestamp: {ts}')